**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/ashworth-2025"], check=True)
    os.chdir("/content/encore/entries/ashworth-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Ashworth et al. (2025), ongoing activity: spectra, regularity, period scan and coherence, on dandiset 001708

**Paper.** Ashworth et al., "Multi-timescale dynamics organize descending pain modulation", bioRxiv
doi:10.64898/2025.12.15.693905, version 4, with its supplement; the claims file
(`claims/001708/10.64898-2025.12.15.693905.yaml`) was written from this text. Method details not in the
claims file come from the paper or from its code (github.com/nox-lab/multi-timescale-rhythmic-dynamics-in-rvm-neurons
at commit e12f3ee771f4b5506e5db0bac09762ba6044a4d6), cited where used.

**Dataset.** DANDI:001708, version 0.260604.1023 (the version the paper cites, and the only
published one), streamed.

| claim | tested here |
|---|---|
| ongoing-cell-counts | yes |
| fig3bc-psd-peaks | yes |
| fig3d-neutral-psd | yes |
| fig3e-cv-isi | yes |
| fig4-nlml-minima | yes |
| fig6c-coherence-counts | yes |
| fig6d-heart-rate-peaks | yes |

The Gaussian-process fits (fig5 claims, Table 1, figs2-phase-latency) are in `gp.ipynb` and `gp_methods.ipynb`.

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import rvm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
T0 = time.time()
print("dandiset", rvm.DANDISET, "version", rvm.VERSION)

## Data check

In [ ]:
assets = [a for a in rvm.list_assets()]
sessions, cells = rvm.load_all(assets)
print(f"loaded {len(sessions)} files in {time.time() - T0:.0f} s")
ong = cells[cells.protocol.isin(["ongoing", "evoked/ongoing"])].copy()
print("cells of the ongoing and evoked/ongoing files:")
print(pd.crosstab(ong.protocol, ong.cls, margins=True))
print("subjects:", ong.groupby("cls").subject.nunique().to_dict(), "files:", ong.groupby("cls").path.nunique().to_dict())
print("recording length (s) by protocol:", ong.groupby("protocol").t_stop.describe()[["min", "max"]].round(1).to_dict("index"))
nekg = {p: len(rvm.ekg(sessions[i])) for i, p in zip(ong.session, ong.path)}
print("ongoing-set files without EKG beats:", [p for p, n in nekg.items() if n == 0])
ibis = np.concatenate([np.diff(rvm.ekg(sessions[i])) for i in ong.session.unique()])
print(f"inter-beat intervals: median {np.median(ibis):.3f} s, 1st and 99th percentiles {np.percentile(ibis, 1):.3f} "
      f"and {np.percentile(ibis, 99):.3f} s, under 50 ms: {(ibis < 0.05).sum()}")
# EKG beats and spikes share the recording's clock range
rng_ok = all(rvm.ekg(sessions[i]).max() <= max(sessions[i]["t_stop"]) + 1 for i in ong.session.unique())
print("EKG beat times within each recording:", rng_ok)

## ongoing-cell-counts

> The ongoing-activity analyses use 25 ON, 25 OFF and 32 NEUTRAL cells recorded without external stimulation.

In [ ]:
SEG = {}
rows = []
for ci, c in ong.iterrows():
    s = sessions[c.session]
    a, b, gap, buf = rvm.ongoing_segment(s, c.t_stop, 1500.0)
    allev = np.concatenate([rvm.heat(s), rvm.withdrawals(s), rvm.pinches(s)])
    inside = allev[(allev > a) & (allev < b)]
    SEG[ci] = (a, b)
    rows.append(dict(ci=ci, path=c.path, protocol=c.protocol, cls=c.cls, start=a, end=b, longest_heat_gap=gap,
                     buffer_after_previous_heat=buf, events_inside=len(inside), ends_at_end=bool(np.isclose(b, c.t_stop))))
seg = pd.DataFrame(rows).set_index("ci")
print("cells per class:", ong.cls.value_counts().to_dict(), "(expected ON 25, OFF 25, NEUTRAL 32)")
oo = seg[seg.protocol == "ongoing"]
print("ongoing files, 1500 s segment ending at the heat onset that closes the longest gap:")
print(oo.groupby("cls")[["longest_heat_gap", "buffer_after_previous_heat"]].describe().round(1).T)
print("segments starting before 0:", int((oo.start < 0).sum()), "; events (heat, withdrawal, pinch) inside a segment:",
      int(oo.events_inside.sum()), "; segments ending at the recording end:", int(oo.ends_at_end.sum()))
print("ON and OFF cells with a 1500 s segment free of heat onsets:",
      int(((oo.cls != "NEUTRAL") & (oo.longest_heat_gap >= 1500) & (oo.start >= 0)).sum()), "of", int((oo.cls != "NEUTRAL").sum()))
print("buffers under the Methods' 230 s:", oo[oo.buffer_after_previous_heat < 230][["path", "cls", "buffer_after_previous_heat"]].round(1).to_string())
print("evoked/ongoing NEUTRAL cells use their whole recordings:", seg[seg.protocol == "evoked/ongoing"].shape[0],
      "cells, lengths", sorted(set(np.round(ong[ong.protocol == "evoked/ongoing"].t_stop, 1))))

def train_seg(ci):
    c = cells.loc[ci]
    a, b = SEG[ci]
    return rvm.cut(sessions[c.session]["trains"][c.unit], a, b), b - a

def train_whole(ci):
    c = cells.loc[ci]
    return sessions[c.session]["trains"][c.unit], c.t_stop

ids = {k: list(ong.index[ong.cls == k]) for k in ["ON", "OFF", "NEUTRAL"]}

**Result: reproduced.** 25 ON, 25 OFF and 32 NEUTRAL cells; all 50 ON and OFF cells have a 1500 s segment free of events. Four ON cells in two files have a buffer of 226.5 to 228.6 s, under the Methods' 230 s.

## fig3bc-psd-peaks

> ON and OFF cells show clear low-frequency spectral peaks, often with harmonics, at a period of about
> five minutes (ON 286.86 s, FWHM 16.92 s; OFF 298.33 s, FWHM 16.12 s).

In [ ]:
first_on = train_seg(ids["ON"][0])[0]
NFFT = 50 * len(first_on)
print("FFT length (50 times the spike count of the first ON cell, as the code):", NFFT,
      f"; resolution {1 / (NFFT * 0.005):.3e} Hz")
PSD = {}
for k in ["ON", "OFF"]:
    PSD[k] = [rvm.spike_psd(*train_seg(ci), NFFT) for ci in ids[k]]
PSD["NEUTRAL"] = [rvm.spike_psd(*train_whole(ci), NFFT) for ci in ids["NEUTRAL"]]
F = PSD["ON"][0][0]
MEAN = {k: np.mean([p[1] for p in v], axis=0) for k, v in PSD.items()}
HF_F = PSD["ON"][0][2]
HF = {k: np.mean([p[3] for p in v], axis=0) for k, v in PSD.items()}
print(f"PSD computed for {sum(len(v) for v in PSD.values())} cells; low-frequency part 0 to {F[-1]:.3f} Hz, {len(F)} bins")

In [ ]:
def fwhm_seconds(fp, w_hz):
    return dict(a=w_hz / fp ** 2, b=1 / (fp - w_hz / 2) - 1 / (fp + w_hz / 2))

res = {}
for k in ["ON", "OFF", "NEUTRAL"]:
    i, _ = rvm.highest_peak(F, MEAN[k], 1e-4, 0.1)
    fp = F[i]
    # per-cell search band: 0.5 to 1.95 times the class peak; for NEUTRAL, the ON peak (criterion of fig3d-neutral-psd)
    f_band = res["ON"]["f_peak"] if k == "NEUTRAL" else fp
    per_cell, per_cell_code, cell_peaks = [], [], []
    for (f, p, _, _) in PSD[k]:
        j, jm = rvm.highest_peak(f, p, 0.5 * f_band, 1.95 * f_band)
        if j is None:
            per_cell.append(np.nan); per_cell_code.append(np.nan); cell_peaks.append(np.nan)
            continue
        per_cell.append(rvm.fwhm_hz(f, p, j))
        per_cell_code.append(rvm.fwhm_hz(f, p, jm))  # the code passes the index within the masked range
        cell_peaks.append(f[j])
    w_mean = rvm.fwhm_hz(F, MEAN[k], i)
    res[k] = dict(f_peak=fp, period=1 / fp, f_band=f_band, fwhm_cells_hz=np.nanmean(per_cell), fwhm_cells_code_hz=np.nanmean(per_cell_code),
                  fwhm_mean_psd_hz=w_mean, n_cells=len(PSD[k]), n_no_peak=int(np.isnan(per_cell).sum()),
                  cell_periods_median=np.nanmedian(1 / np.array(cell_peaks)))
    r = res[k]
    sa = fwhm_seconds(fp, r["fwhm_cells_hz"]); sc = fwhm_seconds(fp, r["fwhm_mean_psd_hz"]); sk = fwhm_seconds(fp, r["fwhm_cells_code_hz"])
    print(f"{k}: class-mean peak {fp:.6f} Hz, period {1 / fp:.2f} s; FWHM per cell (mean of {r['n_cells'] - r['n_no_peak']} cells) "
          f"{r['fwhm_cells_hz']:.3e} Hz -> (a) {sa['a']:.2f} s, (b) {sa['b']:.2f} s; mean-PSD FWHM {r['fwhm_mean_psd_hz']:.3e} Hz -> "
          f"(c) {sc['b']:.2f} s; code's index: {r['fwhm_cells_code_hz']:.3e} Hz -> (a) {sk['a']:.2f} s, (b) {sk['b']:.2f} s; "
          f"median per-cell peak period {r['cell_periods_median']:.1f} s")
print("expected: ON 286.86 +- 28.69 s, FWHM 16.92 +- 1.69 s; OFF 298.33 +- 29.83 s, FWHM 16.12 +- 1.61 s")
for k in ["ON", "OFF"]:
    # harmonics and subharmonics: local maxima of the class mean near 600 s and 150 s
    pk, _ = find_peaks(MEAN[k])
    near = {T: [round(1 / F[j], 1) for j in pk if F[j] > 0 and abs(1 / F[j] - T) <= 0.1 * T and MEAN[k][j] > np.median(MEAN[k][(F > 1e-3) & (F < 1e-2)])]
            for T in (600, 150)}
    print(f"{k}: local maxima within 10% of 600 s and 150 s above the median 0.001-0.01 Hz power: {near}")

In [ ]:
# Reading of the supplement ("Spike train power spectra", paragraph 3): hierarchical bootstrap at
# the rat level, N rats resampled with replacement with all their cells, mean PSD per iteration,
# its highest peak (1e-4 to 0.1 Hz) and that peak's FWHM, 1000 iterations.
rng = np.random.default_rng(301)
boot = {}
band = (F >= 1e-4) & (F <= 0.1)
for k in ["ON", "OFF"]:
    subj = ong.loc[ids[k], "subject"].values
    animals = np.unique(subj)
    P = np.array([p[1] for p in PSD[k]])
    per, fw = [], []
    for b in range(1000):
        pick = rng.choice(animals, size=len(animals), replace=True)
        rows_ = np.concatenate([np.flatnonzero(subj == a) for a in pick])
        m = P[rows_].mean(0)
        i, _ = rvm.highest_peak(F, m, 1e-4, 0.1)
        per.append(1 / F[i]); fw.append(rvm.fwhm_hz(F, m, i) if i is not None else np.nan)
        fpb = F[i]
    per, fw = np.array(per), np.array(fw)
    fp_mean = 1 / per.mean()
    boot[k] = dict(period=per.mean(), period_sd=per.std(), fwhm_hz=np.nanmean(fw))
    print(f"{k} bootstrap ({len(animals)} rats): period mean {per.mean():.2f} s (sd {per.std():.2f}, 2.5-97.5% "
          f"{np.percentile(per, 2.5):.1f} to {np.percentile(per, 97.5):.1f}); FWHM mean {np.nanmean(fw):.3e} Hz -> "
          f"(a) {np.nanmean(fw) / fp_mean ** 2:.2f} s, (b) {1 / (fp_mean - np.nanmean(fw) / 2) - 1 / (fp_mean + np.nanmean(fw) / 2):.2f} s")

In [ ]:
# The code's stored results (fwhm_results_*.pkl) need the code's own environment to unpickle; the
# values printed by the claims file are 288.9 s and 303.3 s and FWHMs of 2.13e-4 and 1.23e-4 Hz.
fig, axs = plt.subplots(1, 3, figsize=(12, 3), sharey=False)
for ax, k in zip(axs, ["ON", "OFF", "NEUTRAL"]):
    m = (F > 5e-4) & (F < 0.02)
    ax.plot(F[m], MEAN[k][m], "k", lw=0.8)
    ax.axvline(1 / 300, color="tab:red", lw=0.5, ls="--")
    ax.set_title(k); ax.set_xlabel("frequency (Hz)")
    print(k, "plotted PSD range", f"{MEAN[k][m].min():.3g} to {MEAN[k][m].max():.3g}", "over", f"{F[m][0]:.4f} to {F[m][-1]:.4f} Hz")
axs[0].set_ylabel("normalized power")
plt.tight_layout(); plt.show()

**Result: partially reproduced.** The class-mean peaks are at 286.86 s (ON) and 298.33 s (OFF), the paper's values. No reading gives the paper's FWHMs of 16.92 and 16.12 s: per-cell widths give 90.2 and 102.7 s, the class mean 83.9 and 104.2 s, the code's index 10.2 and 8.0 s, the rat bootstrap 57.8 and 97.9 s.

## fig3d-neutral-psd

> NEUTRAL cells lack consistent low-frequency spectral peaks and instead show broad low-frequency
> components and prominent power above 10 Hz, reflecting regular tonic firing.

"Local maxima above 10 Hz that the ON and OFF mean PSDs lack" is read on the spectra averaged in
0.5 Hz bins (a standard smoothing choice): a local maximum above 10 Hz whose prominence exceeds 10%
of the class's median power over 10 to 100 Hz (a threshold the claims file does not give; an
assumption, shown with other thresholds).

In [ ]:
lowband = (F >= 1e-3) & (F <= 1e-2)
win = (F >= 0.0025) & (F <= 0.0045)
ratio = {}
for k in ["ON", "OFF", "NEUTRAL"]:
    pk, _ = find_peaks(MEAN[k])
    pk = pk[win[pk]]
    med = np.median(MEAN[k][lowband])
    ratio[k] = MEAN[k][pk].max() / med if len(pk) else 0.0
    print(f"{k}: highest local maximum of the mean PSD between 0.0025 and 0.0045 Hz over the 0.001-0.01 Hz median: {ratio[k]:.2f}")
print(f"criterion: NEUTRAL ratio {ratio['NEUTRAL']:.2f} must not exceed ON ratio / 2 = {ratio['ON'] / 2:.2f}: "
      f"{ratio['NEUTRAL'] <= ratio['ON'] / 2}")
i, _ = rvm.highest_peak(F, MEAN["NEUTRAL"], 1e-4, 0.1)
print(f"NEUTRAL class-mean highest peak 1e-4 to 0.1 Hz: {F[i]:.3e} Hz (period {1 / F[i]:.0f} s); per-cell peak periods "
      f"between 0.5 and 1.95 times the ON peak frequency ({res['NEUTRAL']['f_band']:.3e} Hz, {1 / res['NEUTRAL']['f_band']:.1f} s; "
      f"band {1 / (1.95 * res['NEUTRAL']['f_band']):.0f} to {1 / (0.5 * res['NEUTRAL']['f_band']):.0f} s): median "
      f"{res['NEUTRAL']['cell_periods_median']:.1f} s over {res['NEUTRAL']['n_cells'] - res['NEUTRAL']['n_no_peak']} cells with a peak in the band")
for k in ["ON", "OFF"]:
    print(f"  {k} per-cell peak periods in the same kind of band (their own class peak): median {res[k]['cell_periods_median']:.1f} s")
from scipy.signal import find_peaks as fp_
hfm = HF_F > 10
for thr in [0.05, 0.1, 0.2]:
    out = {}
    for k in ["ON", "OFF", "NEUTRAL"]:
        h = HF[k][hfm]
        pk, pr = fp_(h, prominence=thr * np.median(h))
        out[k] = [round(x, 1) for x in HF_F[hfm][pk]][:6]
    print(f"local maxima above 10 Hz with prominence > {thr:.0%} of the median 10-100 Hz power: "
          + "; ".join(f"{k}: {len(v)} {v}" for k, v in out.items()))
for k in ["ON", "OFF", "NEUTRAL"]:
    tot = np.trapezoid(HF[k], HF_F)
    print(f"{k}: fraction of power above 10 Hz {np.trapezoid(HF[k][hfm], HF_F[hfm]) / tot:.3f}; "
          f"max/median of the 10-100 Hz spectrum {HF[k][hfm].max() / np.median(HF[k][hfm]):.2f}")
fig, ax = plt.subplots(figsize=(6, 3))
for k, col in [("ON", "tab:red"), ("OFF", "tab:blue"), ("NEUTRAL", "0.4")]:
    ax.semilogy(HF_F[1:], HF[k][1:], color=col, lw=0.8, label=k)
ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("normalized power (0.5 Hz bins)"); ax.legend()
print("plotted range", {k: (f"{HF[k][1:].min():.2e}", f"{HF[k][1:].max():.2e}") for k in HF})
plt.show()

**Result: reproduced, resting on an assumption.** The NEUTRAL spectrum has no peak near 300 s comparable to the ON one (ratio 1.43 against 7.82), and above 10 Hz it has local maxima (11.8, 17.8, 33.8 Hz at a 10% prominence threshold) that ON and OFF lack; at a 5% threshold all three classes have some. Searched between 0.5 and 1.95 times the ON peak frequency (147 to 574 s), the NEUTRAL per-cell peaks have a median period of 271.3 s.

## fig3e-cv-isi

> NEUTRAL cells fire regularly and fast (CV 0.35 ± 0.26, mean ISI 0.17 ± 0.25 s), while ON and OFF
> cells fire irregularly, with CVs well above 1 (ON 4.91 ± 4.32, OFF 3.70 ± 3.09) and broad ISI
> distributions (ON 1.25 ± 4.80 s, OFF 0.84 ± 1.36 s).

In [ ]:
rows = []
for ci, c in ong.iterrows():
    tr, _ = train_whole(ci)
    isi = np.diff(tr)
    rows.append(dict(cls=c.cls, cv=isi.std() / isi.mean(), mean_isi=isi.mean()))
cvt = pd.DataFrame(rows).groupby("cls").agg(CV_mean=("cv", "mean"), CV_sd=("cv", "std"), ISI_mean=("mean_isi", "mean"),
                                             ISI_sd=("mean_isi", "std"), n=("cv", "size"))
print(cvt.round(3))
exp_ = {"NEUTRAL": (0.348, 0.264, 0.165, 0.25), "OFF": (3.699, 3.087, 0.835, 1.357), "ON": (4.909, 4.319, 1.248, 4.799)}
for k, (cv, cvs, isi, isis) in exp_.items():
    r = cvt.loc[k]
    print(f"{k}: CV {r.CV_mean:.3f} vs {cv} (within spread {abs(r.CV_mean - cv) <= cvs}, within 10% {abs(r.CV_mean - cv) <= 0.1 * cv}); "
          f"mean ISI {r.ISI_mean:.3f} vs {isi} (within spread {abs(r.ISI_mean - isi) <= isis}, within 10% {abs(r.ISI_mean - isi) <= 0.1 * isi})")
print("NEUTRAL mean CV below 1:", cvt.loc["NEUTRAL"].CV_mean < 1, "; ON and OFF above 1:", bool((cvt.loc[["ON", "OFF"]].CV_mean > 1).all()))

**Result: reproduced.** CV 0.348, 3.699 and 4.909 and mean ISI 0.165, 0.835 and 1.248 s for NEUTRAL, OFF and ON, the paper's values to the third decimal.

## fig4-nlml-minima

> Scans of a periodic Gaussian process over the period show clear minima of the negative log marginal
> likelihood near 300 s for ON and OFF cells, with further minima at 600 and 900 s, while NEUTRAL
> cells, with one exception, have flat profiles without consistent minima.

In [ ]:
PERIODS = np.linspace(0.1, 1500, 200)
PERIODS500 = np.linspace(0.1, 500, 200)
def scan(ci, whole_neutral=True, remove=2, periods=PERIODS):
    c = cells.loc[ci]
    if c.cls == "NEUTRAL" and (whole_neutral or c.protocol == "evoked/ongoing"):
        tr, T = train_whole(ci)
    else:
        tr, T = train_seg(ci)
    t, r = rvm.smoothed_rate(tr, T)
    t, r = t[remove:len(t) - remove], r[remove:len(r) - remove]
    return rvm.nlml_scan(t, r, periods)

t1 = time.time()
NL = {ci: scan(ci) for ci in ong.index}
print(f"scan of {len(NL)} cells at 200 periods: {time.time() - t1:.0f} s")
NL_seg_neut = {ci: scan(ci, whole_neutral=False) for ci in ids["NEUTRAL"] if cells.loc[ci, "protocol"] == "ongoing"}
NL_r1 = {ci: scan(ci, remove=1) for ci in ids["ON"] + ids["OFF"]}
NL500 = {ci: scan(ci, periods=PERIODS500) for ci in ids["ON"] + ids["OFF"]}

In [ ]:
def minima_near(prof, periods, p, tol=0.1):
    from scipy.signal import argrelmin
    j = argrelmin(prof)[0]
    return bool(np.any(np.abs(periods[j] - p) <= tol * p))

def summary(nl, label, periods=PERIODS, targets=(300, 600, 900)):
    out = {}
    for k in ["ON", "OFF", "NEUTRAL"]:
        cis = [ci for ci in ids[k] if ci in nl]
        if not cis:
            continue
        norm = np.array([nl[ci] / np.max(nl[ci]) for ci in cis])
        cnt = {p: int(sum(minima_near(r, periods, p) for r in norm)) for p in targets}
        mean_prof = norm.mean(0)
        cm = {p: minima_near(mean_prof, periods, p) for p in targets}
        argmin = periods[np.argmin(norm, axis=1)]
        out[k] = dict(n=len(cis), cells_with_min=cnt, class_mean_min=cm, median_argmin=float(np.median(argmin)))
        print(f"{label} {k}: {len(cis)} cells; cells with a local minimum within 10% of {list(cnt)} s: {list(cnt.values())}; "
              f"class mean has minima there: {list(cm.values())}; median global argmin {np.median(argmin):.1f} s")
    return out

S_main = summary(NL, "main (NEUTRAL whole, 2 samples removed)")
ok_onoff = all(S_main[k]["cells_with_min"][300] > S_main[k]["n"] / 2 and all(S_main[k]["class_mean_min"].values()) for k in ["ON", "OFF"])
print("majority of ON and of OFF cells with a minimum near 300 s, and class means with minima near 300, 600 and 900 s:", ok_onoff)
print("NEUTRAL cells with a minimum near 300 s:", S_main["NEUTRAL"]["cells_with_min"][300], "(at most 4 for the claim)")
summary(NL_r1, "variant (ON and OFF, one sample removed at each end, the Methods' first and last 2 s)")
summary(NL_seg_neut, "variant (ongoing NEUTRAL cells cut to 1500 s, as the code's stored scan)")
summary(NL500, "variant (periods 0.1 to 500 s, the Methods)", periods=PERIODS500, targets=(300,))

In [ ]:
stored = np.load(rvm.code_file("notebooks/gpytorch_models/PERIOD_SCOPE_RESULTS_200.npz"))
print("stored scan arrays:", {k: stored[k].shape for k in stored.files})
print("period grid identical:", np.allclose(stored["param_values"], PERIODS))
for k, mine in [("ON", ids["ON"]), ("OFF", ids["OFF"])]:
    st = stored[k].T
    r = [np.corrcoef(st[i], NL[ci])[0, 1] for i, ci in enumerate(mine)]
    am_st, am = PERIODS[np.argmin(st, 1)], PERIODS[np.argmin([NL[ci] for ci in mine], 1)]
    print(f"{k}: correlation of each cell's NLML profile with the stored one, in file order: median {np.median(r):.3f} "
          f"(min {np.min(r):.3f}); same global argmin: {int(np.sum(np.isclose(am_st, am)))} of {len(mine)}; "
          f"stored median argmin {np.median(am_st):.1f} s, here {np.median(am):.1f} s")
# the stored profiles are in the code's file order, which is not the archive's; each stored
# profile is matched to the cell of this scan whose profile it correlates with best
for k, mine, keys in [("ON", ids["ON"], ["ON"]), ("OFF", ids["OFF"], ["OFF"]),
                      ("NEUTRAL", ids["NEUTRAL"], ["NEUTRAL", "NEUTRAL_extra"])]:
    st = np.hstack([stored[q] for q in keys]).T
    pool = {ci: NL[ci] for ci in mine}
    pool.update({("seg", ci): v for ci, v in NL_seg_neut.items()} if k == "NEUTRAL" else {})
    best = [max(np.corrcoef(row, v)[0, 1] for v in pool.values()) for row in st]
    print(f"{k}: best correlation of each of the {len(st)} stored profiles with a profile here: median {np.median(best):.4f}, "
          f"min {np.min(best):.4f}; above 0.99: {int(np.sum(np.array(best) > 0.99))}")
fig, axs = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, k in zip(axs, ["ON", "OFF", "NEUTRAL"]):
    norm = np.array([NL[ci] / NL[ci].max() for ci in ids[k]])
    ax.plot(PERIODS, norm.T, color="0.7", lw=0.4); ax.plot(PERIODS, norm.mean(0), "k", lw=1.2)
    for p in (300, 600, 900):
        ax.axvline(p, color="tab:red", lw=0.5, ls="--")
    ax.set_title(k); ax.set_xlabel("period (s)")
    print(k, "normalized NLML range", round(norm.min(), 3), "to", round(norm.max(), 3))
axs[0].set_ylabel("NLML / max"); plt.tight_layout(); plt.show()

**Result: partially reproduced.** 25 of 25 ON and 23 of 25 OFF cells have a minimum near 300 s and the class means have minima near 300, 600 and 900 s, but so do 25 of 32 NEUTRAL cells. The scan matches the code's stored scan (each stored profile correlates at 1.0000 with one here).

## fig6c-coherence-counts

> A subset of cells of every class shows significant low-frequency coherence with heart rate (5 of 32
> NEUTRAL, 4 of 25 OFF, 8 of 25 ON).

In [ ]:
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
def coh_cell(ci, reading, seed=1, clean=False):
    c = cells.loc[ci]
    s = sessions[c.session]
    tr = s["trains"][c.unit]
    if reading == "segment" and not (c.cls == "NEUTRAL" and c.protocol == "evoked/ongoing"):
        a, b = SEG[ci]
    else:
        a, b = 0.0, c.t_stop
    beats = rvm.clean_beats(rvm.ekg(s)) if clean else rvm.ekg(s)
    return rvm.coherence_test(tr, beats, a, b, seed=seed)

t1 = time.time()
COH = {}
rows = []
for reading, clean in [("segment", False), ("whole", False), ("segment, EKG cleaned", True)]:
    for ci in ong.index:
        f, C, thr, nseg, thr_pos = coh_cell(ci, reading.split(",")[0], clean=clean)
        COH[(reading, ci)] = (f, C, thr)
        sig = (C > thr) & (f > 0)
        sig_low = sig & (f < 0.01)
        rows.append(dict(reading=reading, ci=ci, cls=cells.loc[ci, "cls"], nseg=nseg, thr=thr, thr_pos=thr_pos, sig=bool(sig.any()),
                         sig_low=bool(sig_low.any()), sig_with_f0=bool((C > thr).any()), sig_pos=bool(((C > thr_pos) & (f > 0)).any()),
                         f_peak=float(f[1:][np.argmax(C[1:])]), cmax=float(C[1:].max())))
coh = pd.DataFrame(rows)
print(f"coherence for {len(ong)} cells under three readings: {time.time() - t1:.0f} s")
tab = coh.groupby(["reading", "cls"]).agg(n=("sig", "size"), significant=("sig", "sum"), significant_below_001Hz=("sig_low", "sum"),
                                          segments_min=("nseg", "min"), segments_max=("nseg", "max"), threshold_median=("thr", "median"))
print(tab)
print("expected: NEUTRAL 5 of 32, OFF 4 of 25, ON 8 of 25 (exact)")
print("frequency-range variants (main: threshold from the surrogates' maximum over all f, a cell counted at f > 0):")
print(coh.groupby(["reading", "cls"]).agg(main=("sig", "sum"), f0_counted_too=("sig_with_f0", "sum"),
                                          f0_left_out_of_both=("sig_pos", "sum"), threshold_median=("thr", "median"),
                                          threshold_f_gt_0_median=("thr_pos", "median")))

In [ ]:
rows = []
for seed in [2, 3, 4]:
    for ci in ong.index:
        f, C, thr, nseg, _ = coh_cell(ci, "segment", seed=seed)
        rows.append(dict(seed=seed, cls=cells.loc[ci, "cls"], sig=bool(((C > thr) & (f > 0)).any())))
print("segment reading, other surrogate seeds:")
print(pd.DataFrame(rows).groupby(["seed", "cls"]).sig.sum().unstack())

**Result: partially reproduced.** On the ongoing segments 3 of 32 NEUTRAL, 6 of 25 OFF and 9 of 25 ON cells are coherent with heart rate, each within 2 of the paper's 5, 4 and 8; over whole recordings 6, 5 and 12. The counts move with the choices: at other surrogate seeds OFF is 6 or 7, leaving the f = 0 bin out of the surrogate maximum gives 5, 7 and 9, and removing the EKG double detections gives 4, 7 and 13.

## fig6d-heart-rate-peaks

> Peak coherence frequencies do not match the ~300 s period exactly, but the heart-rate power spectrum
> has peaks at multiples of this period.

In [ ]:
recs = ong.drop_duplicates("path")
hr_psd = {}
for clean in (False, True):
    for reading in ["segment", "whole"]:
        hr_psd[(reading, clean)] = {}
for ci, c in recs.iterrows():
    s = sessions[c.session]
    if len(rvm.ekg(s)) == 0:
        continue
    if c.protocol == "ongoing":
        a, b = SEG[ci]
    else:
        a, b = 0.0, c.t_stop
    for clean in (False, True):
        beats = rvm.clean_beats(rvm.ekg(s)) if clean else rvm.ekg(s)
        hr_psd[("segment", clean)].setdefault(c.subject, []).append(rvm.spike_psd(rvm.cut(beats, a, b), b - a, NFFT)[1])
        hr_psd[("whole", clean)].setdefault(c.subject, []).append(rvm.spike_psd(beats[beats <= c.t_stop], c.t_stop, NFFT)[1])
targets = {"subharmonic 600 s": 600, "subharmonic 900 s": 900, "harmonic 150 s": 150, "harmonic 100 s": 100}
band = (F > 0) & (F < 0.01)

def top_peak(m):
    pk, _ = find_peaks(m)
    pk = pk[band[pk]]
    return pk, pk[np.argmax(m[pk])]

def is_multiple(j):
    return any(abs(1 / F[j] - T) <= 0.1 * T for T in targets.values())

for (reading, clean), d in hr_psd.items():
    per_animal = np.array([np.mean(v, axis=0) for v in d.values()])
    m = per_animal.mean(0)
    pk, top = top_peak(m)
    near = {lab: [round(float(1 / F[j]), 1) for j in pk if abs(1 / F[j] - T) <= 0.1 * T] for lab, T in targets.items()}
    order = pk[np.argsort(m[pk])[::-1]][:5]
    lab = f"{reading}{', EKG cleaned' if clean else ''}"
    print(f"{lab}: {len(d)} animals; highest peak below 0.01 Hz at {F[top]:.5f} Hz (period {1 / F[top]:.1f} s); "
          f"it is a multiple of 300 s: {is_multiple(top)}; lowest-frequency peak {1 / F[pk[0]]:.1f} s; "
          f"five highest peaks (s): {[round(float(1 / F[j]), 1) for j in order]}; local maxima near multiples: {near}")
    # the supplement's bootstrap over animals ("in the same manner as for the cells"): animals resampled with
    # replacement, 1000 iterations, the mean spectrum and its highest peak below 0.01 Hz per iteration
    rng = np.random.default_rng(29)
    tops = []
    for _ in range(1000):
        mb = per_animal[rng.integers(0, len(per_animal), len(per_animal))].mean(0)
        tops.append(top_peak(mb)[1])
    tops = np.array(tops)
    per = 1 / F[tops]
    print(f"  bootstrap over animals (1000): highest peak below 0.01 Hz, median period {np.median(per):.1f} s "
          f"(2.5 to 97.5%: {np.percentile(per, 2.5):.1f} to {np.percentile(per, 97.5):.1f} s); fraction of iterations whose "
          f"highest peak is a multiple of 300 s: {np.mean([is_multiple(j) for j in tops]):.3f}; mean period over iterations {per.mean():.1f} s")
cc = coh[(coh.reading == "segment") & coh.sig]
print("peak coherence frequency of the coherent cells (segment reading), against 1/300 =", round(1 / 300, 5), "Hz:")
print(cc.groupby("cls").f_peak.describe().round(4))

**Result: not reproduced.** The highest peak of the heart-beat spectrum below 0.01 Hz is at 1243 s (segments) or 2486 s (whole recordings), not at a multiple of 300 s; peaks at 286.9 s and near 150 s are present but lower. With double detections removed the highest peaks are at 1491.7 and 2486.1 s, and over whole recordings local maxima at 932.3 and 621.5 s appear, lower. In the bootstrap over animals the highest peak is at a multiple of 300 s in at most 27% of iterations. Coherence peaks (median 0.010 to 0.014 Hz) do not match 1/300 Hz. The rule is asked of a reader.

## Summary

| claim | outcome |
|---|---|
| ongoing-cell-counts | reproduced |
| fig3bc-psd-peaks | partially reproduced |
| fig3d-neutral-psd | reproduced (assumed) |
| fig3e-cv-isi | reproduced |
| fig4-nlml-minima | partially reproduced |
| fig6c-coherence-counts | partially reproduced |
| fig6d-heart-rate-peaks | not reproduced |

How the analysis differs from the paper's. Spectra, period scan and coherence follow the code; the period scan
is the code's to the precision shown. The FWHM in seconds and the rule for peaks above 10 Hz are readings the paper
does not fix. No code exists for the heart-rate spectrum of Figure 6d. Coherence and the heart-rate spectrum are also shown with EKG double detections removed, a variant.

What was found about the dataset. The ongoing segments exist for every ON and OFF cell; four ON cells have a
buffer slightly under the Methods' 230 s. About 2% of EKG inter-beat intervals are under 50 ms, double detections.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

In [ ]:
print(f"notebook runtime {(time.time() - T0) / 60:.1f} min")